# 의존성으로 Agent 주입

`Depends`가 라우트와 Agent 생성 책임을 어떻게 분리하는지 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

## 코드 읽기와 개념 경계

`Depends(get_agent)`는 `get_agent`를 지금 직접 호출한 결과가 아니라, 요청 시 FastAPI가 해결할 제공자 함수로 등록합니다. 라우트는 제공자가 반환한 Agent의 `ainvoke`만 사용합니다. 테스트의 `app.dependency_overrides[get_agent] = ...`는 원래 제공자를 같은 역할의 `StubAgent`로 바꾸고, HTTP 경로와 요청 검증은 그대로 통과시킵니다. 따라서 출력의 `대체 응답`은 라우트 전체를 가짜로 만든 결과가 아니라 Agent 경계만 교체한 결과입니다.

설정처럼 읽기 전용인 값은 여러 요청이 공유해도 되지만, 데이터베이스 세션처럼 닫아야 하는 자원은 요청마다 만들고 정리해야 합니다. 대체 객체가 성공했다는 사실도 실제 모델 품질을 보장하지 않습니다. 의존성 테스트는 연결 방식과 응답 변환을 확인하고, 실제 Agent 통합 테스트는 도구 선택과 생성 결과를 별도로 확인합니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 기본 실행

운영 객체 대신 테스트 객체를 의존성 재정의 표에 넣습니다.

In [2]:
from fastapi.testclient import TestClient
from agent_service.app import app, get_agent

class StubAgent:
    async def ainvoke(self, state):
        return {'answer': '대체 응답'}

app.dependency_overrides[get_agent] = lambda: StubAgent()
client = TestClient(app)
headers = {'Authorization': 'Bearer local-demo-token'}

## 대체 결과 관찰

라우트 코드는 그대로인데 응답을 만드는 객체만 바뀝니다.

In [3]:
response = client.post('/v1/chat', headers=headers,
    json={'question': '질문', 'thread_id': 'dep-1'})
print(response.json())
assert response.json()['answer'] == '대체 응답'

{'answer': '대체 응답', 'thread_id': 'dep-1'}


## 실패 재현

재정의를 해제하지 않으면 다음 실험도 대체 객체를 사용합니다.

In [4]:
still_stub = client.post('/v1/chat', headers=headers,
    json={'question': '다음 질문', 'thread_id': 'dep-2'})
print(still_stub.json()['answer'])
assert still_stub.json()['answer'] == '대체 응답'

대체 응답


## 수정

재정의 표를 비운 뒤 실제 LangGraph 객체가 다시 호출됩니다.

In [5]:
app.dependency_overrides.clear()
real = client.post('/v1/chat', headers=headers,
    json={'question': '다음 질문', 'thread_id': 'dep-2'})
print(real.json()['answer'])
assert real.json()['answer'].startswith('질문을 확인했습니다')

질문을 확인했습니다: 다음 질문
